In [1]:
# Cell 1: Install system dependencies
!apt-get install -y bedtools samtools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
samtools is already the newest version (1.13-4).
bedtools is already the newest version (2.30.0+dfsg-2ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Cell 3: Download hg38 reference genome and index (skipped if exists)
%%bash
mkdir -p /content/drive/MyDrive/ML_Project/data
cd /content/drive/MyDrive/ML_Project/data

if [ ! -f hg38.fa ]; then
    echo "Downloading hg38.fa.gz ..."
    wget -q http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
    echo "Decompressing ..."
    gunzip -k hg38.fa.gz
fi

if [ ! -f hg38.fa.fai ]; then
    echo "Indexing ..."
    samtools faidx hg38.fa
fi

In [4]:
# Cell 4: Parse narrowPeak, center 101bp window on summit, filter boundaries
import pandas as pd
import os

BASE = '/content/drive/MyDrive/ML_Project/data'

# Config Datasets
DATASETS = {
    'SP1': 'sp1_raw_data.narrowPeak.bed',
    'SP2': 'sp2_raw_data.narrowPeak.bed',
    'SP4': 'sp4_raw_data.narrowPeak.bed',
}

COLS = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal', 'pval', 'qval', 'peak']

for name, filename in DATASETS.items():
    input_path = os.path.join(BASE, filename)
    output_path = os.path.join(BASE, f'{name.lower()}_centered_101bp.bed')

    if not os.path.exists(input_path):
        print(f"SKIP {name}: {filename} not found")
        continue

    df = pd.read_csv(input_path, sep='\t', header=None, names=COLS)
    df[['start', 'end', 'peak']] = df[['start', 'end', 'peak']].astype(int)

    df['summit'] = df['start'] + df['peak']
    df['new_start'] = df['summit'] - 50
    df['new_end'] = df['summit'] + 51

    initial = len(df)
    df = df[(df['new_start'] >= 0) & (df['new_end'] > df['new_start'])]

    df[['chrom', 'new_start', 'new_end']].to_csv(output_path, sep='\t', header=False, index=False)
    print(f"{name}: {initial} -> {len(df)} peaks -> {output_path}")

print("Done.")

SP1: 16043 -> 16043 peaks -> /content/drive/MyDrive/ML_Project/data/sp1_centered_101bp.bed
SP2: 11301 -> 11301 peaks -> /content/drive/MyDrive/ML_Project/data/sp2_centered_101bp.bed
SP4: 23574 -> 23574 peaks -> /content/drive/MyDrive/ML_Project/data/sp4_centered_101bp.bed
Done.


In [5]:
# Cell 5: Extract chromosome sizes from FASTA index
%%bash
cd /content/drive/MyDrive/ML_Project/data
cut -f 1,2 hg38.fa.fai > hg38.chrom.sizes
echo "Generated hg38.chrom.sizes"

Generated hg38.chrom.sizes


In [ ]:
# Cell 6: Generate positive/negative FASTA + clean for SP1, SP2, SP4
import os
import subprocess

BASE = "/content/drive/MyDrive/ML_Project/data"
PROTEINS = ['sp1', 'sp2', 'sp4']

# PHASE 1: bedtools getfasta + shuffle (bash)
for protein in PROTEINS:
    bed_in = f"{BASE}/{protein}_centered_101bp.bed"
    fa_pos = f"{BASE}/{protein}_positive_101bp.fasta"
    bed_neg = f"{BASE}/{protein}_negative_101bp.bed"
    fa_neg = f"{BASE}/{protein}_negative_101bp.fasta"

    if not os.path.exists(bed_in):
        print(f"SKIP {protein}: {bed_in} not found")
        continue

    print(f"\n--- {protein.upper()} ---")

    # Extract positive sequences
    subprocess.run([
        'bedtools', 'getfasta',
        '-fi', f'{BASE}/hg38.fa',
        '-bed', bed_in,
        '-fo', fa_pos
    ], check=True)
    print(f"  Positive FASTA saved.")

    # Generate negative coordinates
    with open(bed_neg, 'w') as fout:
        subprocess.run([
            'bedtools', 'shuffle',
            '-i', bed_in,
            '-g', f'{BASE}/hg38.chrom.sizes',
            '-excl', bed_in,
            '-noOverlapping',
            '-seed', '42'
        ], stdout=fout, check=True)
    print(f"  Negative BED saved.")

    # Extract negative sequences
    subprocess.run([
        'bedtools', 'getfasta',
        '-fi', f'{BASE}/hg38.fa',
        '-bed', bed_neg,
        '-fo', fa_neg
    ], check=True)
    print(f"  Negative FASTA saved.")

# PHASE 2: Clean FASTA (remove N, uppercase)
def clean_fasta(input_path, output_path):
    total, valid, discarded = 0, 0, 0
    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        header, seq_parts = None, []
        for line in fin:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    total += 1
                    seq = ''.join(seq_parts).upper()
                    if 'N' not in seq:
                        fout.write(f"{header}\n{seq}\n")
                        valid += 1
                    else:
                        discarded += 1
                header, seq_parts = line, []
            else:
                seq_parts.append(line)
        if header is not None:
            total += 1
            seq = ''.join(seq_parts).upper()
            if 'N' not in seq:
                fout.write(f"{header}\n{seq}\n")
                valid += 1
            else:
                discarded += 1
    return total, valid, discarded

# PHASE 3: Execute cleaning + summary
print("\n" + "="*50)
print("CLEANING PIPELINE")
print("="*50)

summary = {}
for protein in PROTEINS:
    pos_raw = f"{BASE}/{protein}_positive_101bp.fasta"
    neg_raw = f"{BASE}/{protein}_negative_101bp.fasta"
    pos_clean = f"{BASE}/{protein}_positive_101bp_clean.fasta"
    neg_clean = f"{BASE}/{protein}_negative_101bp_clean.fasta"

    if not os.path.exists(pos_raw):
        continue

    print(f"\n--- {protein.upper()} ---")
    _, v_pos, _ = clean_fasta(pos_raw, pos_clean)
    _, v_neg, _ = clean_fasta(neg_raw, neg_clean)

    summary[protein] = (v_pos, v_neg)
    status = "BALANCED" if v_pos == v_neg else f"IMBALANCE: {abs(v_pos-v_neg)}"
    print(f"  Result: pos={v_pos}, neg={v_neg} -> {status}")

print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
total_pos = sum(v[0] for v in summary.values())
total_neg = sum(v[1] for v in summary.values())
for protein, (pos, neg) in summary.items():
    print(f"  {protein.upper()}: pos={pos}, neg={neg}")
print(f"  TOTAL: pos={total_pos}, neg={total_neg}")
print(f"  Combined dataset: {total_pos + total_neg} sequences")


--- SP1 ---
  Positive FASTA saved.
  Negative BED saved.
  Negative FASTA saved.

--- SP2 ---
  Positive FASTA saved.
  Negative BED saved.
  Negative FASTA saved.

--- SP4 ---
  Positive FASTA saved.
  Negative BED saved.
  Negative FASTA saved.

CLEANING PIPELINE

--- SP1 ---
  Result: pos=16043, neg=15210 -> IMBALANCE: 833

--- SP2 ---
  Result: pos=11301, neg=10727 -> IMBALANCE: 574

--- SP4 ---
  Result: pos=23574, neg=22386 -> IMBALANCE: 1188

FINAL SUMMARY
  SP1: pos=16043, neg=15210
  SP2: pos=11301, neg=10727
  SP4: pos=23574, neg=22386
  TOTAL: pos=50918, neg=48323
  Combined dataset: 99241 sequences


In [2]:
# CELL: Complete Data Pipeline v2.0
# Features: Overlap Removal + Flanking Hard Negatives + Strict Balancing
import pandas as pd
import numpy as np
import os
import subprocess

# CONFIG
BASE = "/content/drive/MyDrive/ML_Project/data"
FLANK_SHIFT = 500
WINDOW_HALF = 50
SEED = 42

PROTEINS = ['sp1', 'sp2', 'sp4']
COLS = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal', 'pval', 'qval', 'peak']


# FEATURE 1: CROSS-CLASS OVERLAP REMOVAL
print("=" * 60)
print("FEATURE 1: Cross-Class Overlap Removal")
print("=" * 60)

def load_and_center(filepath, protein_name):
    """Load narrowPeak, center 101bp on summit."""
    df = pd.read_csv(filepath, sep='\t', header=None, names=COLS)
    df[['start', 'end', 'peak']] = df[['start', 'end', 'peak']].astype(int)
    df['summit'] = df['start'] + df['peak']
    df['new_start'] = df['summit'] - WINDOW_HALF
    df['new_end'] = df['summit'] + WINDOW_HALF + 1
    df = df[(df['new_start'] >= 0) & (df['new_end'] > df['new_start'])]
    df['protein'] = protein_name
    return df[['chrom', 'new_start', 'new_end', 'protein']]

# Load all proteins into one DataFrame
all_dfs = []
for protein in PROTEINS:
    filepath = f"{BASE}/{protein}_raw_data.narrowPeak.bed"
    if not os.path.exists(filepath):
        print(f"  WARNING: {filepath} not found, skipping {protein}")
        continue
    df = load_and_center(filepath, protein)
    all_dfs.append(df)
    print(f"  Loaded {protein}: {len(df)} peaks")

merged = pd.concat(all_dfs, ignore_index=True)

# Find all peaks that overlap with peaks from a DIFFERENT protein
# Strategy: for each peak, check if any peak from another protein shares
# the same chromosome and has overlapping coordinates
overlap_mask = np.zeros(len(merged), dtype=bool)

for i, row in merged.iterrows():
    same_chrom = merged['chrom'] == row['chrom']
    diff_protein = merged['protein'] != row['protein']
    coord_overlap = (merged['new_start'] < row['new_end']) & (merged['new_end'] > row['new_start'])
    if (same_chrom & diff_protein & coord_overlap).any():
        overlap_mask[i] = True

# Remove ALL overlapping peaks (keep only exclusive binding sites)
merged_unique = merged[~overlap_mask].copy()

for protein in PROTEINS:
    before = len(merged[merged['protein'] == protein])
    after = len(merged_unique[merged_unique['protein'] == protein])
    print(f"  {protein}: {before} -> {after} (removed {before - after} cross-class overlaps)")

# Save unique BED files
unique_beds = {}
for protein in PROTEINS:
    subset = merged_unique[merged_unique['protein'] == protein][['chrom', 'new_start', 'new_end']]
    bed_path = f"{BASE}/{protein}_unique_101bp.bed"
    subset.to_csv(bed_path, sep='\t', header=False, index=False)
    unique_beds[protein] = bed_path
    print(f"  Saved: {bed_path} ({len(subset)} exclusive peaks)")


# FEATURE 2: FLANKING-BASED HARD NEGATIVES
print("\n" + "=" * 60)
print("FEATURE 2: Flanking-Based Hard Negative Generation")
print("=" * 60)

# We'll create a single shared negative set, to avoid redundancy
# Strategy: shift all unique positive peaks left AND right by FLANK_SHIFT,
# then exclude any region that overlaps with ANY unique peak

all_flanking_parts = []

for protein in PROTEINS:
    bed_unique = unique_beds.get(protein)
    if bed_unique is None:
        continue

    # Extract positive FASTA
    fa_pos = f"{BASE}/{protein}_positive_101bp.fasta"
    subprocess.run(['bedtools', 'getfasta', '-fi', f'{BASE}/hg38.fa',
                    '-bed', bed_unique, '-fo', fa_pos], check=True, capture_output=True)
    print(f"  {protein}: positive FASTA extracted")

    # Load unique BED for flanking
    df = pd.read_csv(bed_unique, sep='\t', header=None, names=['chrom', 'start', 'end'])

    # Shift left
    left = df.copy()
    left['start'] = left['start'] - FLANK_SHIFT
    left['end'] = left['end'] - FLANK_SHIFT
    left = left[left['start'] >= 0]

    # Shift right
    right = df.copy()
    right['start'] = right['start'] + FLANK_SHIFT
    right['end'] = right['end'] + FLANK_SHIFT

    all_flanking_parts.append(left)
    all_flanking_parts.append(right)

# Merge all flanking regions, deduplicate
flanking_all = pd.concat(all_flanking_parts, ignore_index=True)
flanking_all = flanking_all.drop_duplicates()
print(f"\n  Total flanking candidates: {len(flanking_all)}")

# Save temporary flanking BED
flanking_bed = f"{BASE}/flanking_all_candidates.bed"
flanking_out = flanking_all[['chrom', 'start', 'end']].copy()
flanking_out['start'] = flanking_out['start'].astype(int)
flanking_out['end'] = flanking_out['end'].astype(int)
flanking_out.to_csv(flanking_bed, sep='\t', header=False, index=False)

# Exclude any flanking region that overlaps with ANY unique positive peak
all_unique_beds_list = [f"{BASE}/{p}_unique_101bp.bed" for p in PROTEINS]
all_unique_beds_str = " ".join(all_unique_beds_list)

negative_bed = f"{BASE}/negative_hard_101bp.bed"
exclude_cmd = f"bedtools intersect -a {flanking_bed} -b {all_unique_beds_str} -v > {negative_bed}"
subprocess.run(exclude_cmd, shell=True, check=True, capture_output=True)

neg_count = int(subprocess.check_output(
    ['wc', '-l', negative_bed]).decode().strip().split()[0])
print(f"  Hard negatives after collision check: {neg_count}")

# Extract negative FASTA
fa_neg = f"{BASE}/negative_hard_101bp.fasta"
subprocess.run(['bedtools', 'getfasta', '-fi', f'{BASE}/hg38.fa',
                '-bed', negative_bed, '-fo', fa_neg], check=True, capture_output=True)
neg_fasta_count = int(subprocess.check_output(
    ['grep', '-c', '^>', fa_neg]).decode().strip() or 0)
print(f"  Negative FASTA sequences: {neg_fasta_count}")


# FEATURE 3: STRICT DOWNSAMPLING + FINAL MERGE
print("\n" + "=" * 60)
print("FEATURE 3: Strict Downsampling & Final Merge")
print("=" * 60)

def clean_fasta(input_path, output_path):
    """Remove sequences with N, standardize to uppercase, validate length."""
    total, valid, discarded = 0, 0, 0
    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        header, seq_parts = None, []
        for line in fin:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    total += 1
                    seq = ''.join(seq_parts).upper()
                    if 'N' not in seq and len(seq) == 2 * WINDOW_HALF + 1:
                        fout.write(f"{header}\n{seq}\n")
                        valid += 1
                    else:
                        discarded += 1
                header, seq_parts = line, []
            else:
                seq_parts.append(line)
        if header is not None:
            total += 1
            seq = ''.join(seq_parts).upper()
            if 'N' not in seq and len(seq) == 2 * WINDOW_HALF + 1:
                fout.write(f"{header}\n{seq}\n")
                valid += 1
            else:
                discarded += 1
    return total, valid, discarded

def count_fasta(filepath):
    """Count sequences in a FASTA file."""
    if not os.path.exists(filepath):
        return 0
    result = subprocess.run(['grep', '-c', '^>', filepath], capture_output=True, text=True)
    return int(result.stdout.strip() or 0)

def subsample_fasta(input_path, output_path, target_count, seed=SEED):
    """Randomly select target_count sequences from FASTA."""
    with open(input_path, 'r') as f:
        content = f.read().strip().split('\n')

    entries = []
    i = 0
    while i < len(content):
        if content[i].startswith('>'):
            header = content[i]
            seq = content[i+1] if i + 1 < len(content) else ''
            if len(seq) == 2 * WINDOW_HALF + 1:
                entries.append((header, seq))
            i += 2
        else:
            i += 1

    if len(entries) <= target_count:
        with open(output_path, 'w') as fout:
            for h, s in entries:
                fout.write(f"{h}\n{s}\n")
        return len(entries)

    np.random.seed(seed)
    selected = np.random.choice(len(entries), target_count, replace=False)

    with open(output_path, 'w') as fout:
        for idx in sorted(selected):
            fout.write(f"{entries[idx][0]}\n{entries[idx][1]}\n")
    return target_count

# Clean all positive FASTA files
print("\nCleaning positive sequences...")
pos_counts = {}
for protein in PROTEINS:
    raw_fa = f"{BASE}/{protein}_positive_101bp.fasta"
    clean_fa = f"{BASE}/{protein}_positive_clean.fasta"
    if not os.path.exists(raw_fa):
        continue
    _, valid, _ = clean_fasta(raw_fa, clean_fa)
    pos_counts[protein] = valid
    print(f"  {protein}: {valid} clean sequences")

# Clean negative FASTA
print("\nCleaning negative sequences...")
neg_raw = f"{BASE}/negative_hard_101bp.fasta"
neg_clean = f"{BASE}/negative_hard_clean.fasta"
_, neg_valid, _ = clean_fasta(neg_raw, neg_clean)
print(f"  Negative: {neg_valid} clean sequences")

# Determine minimum class size
class_sizes = list(pos_counts.values()) + [neg_valid]
min_class_size = min(class_sizes)
print(f"\n  Minimum class size: {min_class_size}")
print(f"  Downsampling all classes to: {min_class_size}")

# Create final balanced dataset
final_dir = f"{BASE}/final_dataset"
os.makedirs(final_dir, exist_ok=True)

final_counts = {}

# Downsample each positive class
for protein, count in pos_counts.items():
    clean_fa = f"{BASE}/{protein}_positive_clean.fasta"
    final_fa = f"{final_dir}/{protein}_positive_final.fasta"
    n = subsample_fasta(clean_fa, final_fa, min_class_size)
    final_counts[f"{protein.upper()}"] = n
    print(f"  {protein.upper()}: {count} -> {n}")

# Downsample negative class
neg_final = f"{final_dir}/negative_final.fasta"
n_neg = subsample_fasta(neg_clean, neg_final, min_class_size)
final_counts["Negative"] = n_neg
print(f"  Negative: {neg_valid} -> {n_neg}")

# Print final summary
print("\n" + "=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
total = 0
for cls, count in final_counts.items():
    print(f"  {cls}: {count}")
    total += count
print(f"  {'─' * 20}")
print(f"  TOTAL: {total}")
print(f"  Classes: {len(final_counts)}")
print(f"  Balance: {min_class_size} per class (perfect)")
print(f"\n  Saved to: {final_dir}/")

FEATURE 1: Cross-Class Overlap Removal
  Loaded sp1: 16043 peaks
  Loaded sp2: 11301 peaks
  Loaded sp4: 23574 peaks
  sp1: 16043 -> 6650 (removed 9393 cross-class overlaps)
  sp2: 11301 -> 4830 (removed 6471 cross-class overlaps)
  sp4: 23574 -> 13562 (removed 10012 cross-class overlaps)
  Saved: /content/drive/MyDrive/ML_Project/data/sp1_unique_101bp.bed (6650 exclusive peaks)
  Saved: /content/drive/MyDrive/ML_Project/data/sp2_unique_101bp.bed (4830 exclusive peaks)
  Saved: /content/drive/MyDrive/ML_Project/data/sp4_unique_101bp.bed (13562 exclusive peaks)

FEATURE 2: Flanking-Based Hard Negative Generation
  sp1: positive FASTA extracted
  sp2: positive FASTA extracted
  sp4: positive FASTA extracted

  Total flanking candidates: 50080
  Hard negatives after collision check: 47210
  Negative FASTA sequences: 47210

FEATURE 3: Strict Downsampling & Final Merge

Cleaning positive sequences...
  sp1: 6650 clean sequences
  sp2: 4830 clean sequences
  sp4: 13562 clean sequences

Clean